# WELCOME TO CRESENCIO'S LAB 9: 
# AUTOMATING DATA COLLECTION (VNS)

**Set Up the Environment for Web Scrapping**

In [1]:
import sys
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])

# This version handles spaces in file paths better than the ! exclamation syntax
python = sys.executable

# Register the kernel
subprocess.check_call([python, '-m', 'ipykernel', 'install', '--user', '--name', 'web_scraping_env'])

# Install libraries
subprocess.check_call([python, '-m', 'pip', 'install', 'notebook', 'ipykernel', 'requests', 'beautifulsoup4', 'lxml'])

print("Setup complete!")

Installed kernelspec web_scraping_env in /home/Centaurids/.local/share/jupyter/kernels/web_scraping_env
Setup complete!


In [2]:
# TRY IT — Setup: Verify all libraries are properly installed

import requests, bs4, lxml, pandas
print("All libraries successfully imported!")

All libraries successfully imported!


**START OF PROCEDURE FOR LAB 9**

In [3]:
# PROCEDURE 1: Send an HTTP GET Request

import requests

# Step 1: Define the target URL
url = "https://towardsdatascience.com/"

# Step 2: Define a User-Agent header — essential for ethical and successful scraping
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Step 3: Send the GET request with the specified headers
response = requests.get(url, headers=headers)

# Step 4: Verify the server's response status (200 = success)
print("--- Initiating Contact ---")
print(f"Status Code: {response.status_code}")

# Step 5: Peek at the raw HTML received
print("HTML Content Preview (First 500 characters):")
print(response.text[:500])
print("---")

--- Initiating Contact ---
Status Code: 200
HTML Content Preview (First 500 characters):
<!DOCTYPE html>
<html lang="en-US">
<head>
	<meta charset="UTF-8" />
	<script src="https://h030.towardsdatascience.com/script.js"></script><!-- Google Tag Manager -->
<script>
	(function (w, d, s, l, i) {
		w[l] = w[l] || [];
		w[l].push({
			'gtm.start': new Date().getTime(),
			event: 'gtm.js'
		});
		var f = d.getElementsByTagName(s)[0],
			j = d.createElement(s),
			dl = l != 'dataLayer' ? '&l=' + l : '';
		j.async = true;
		j.src =
			'https://www.googletagmanager.com/gtm.js?id=' + i + dl;

---


In [4]:
# TRY IT — Procedure 1: Try a different URL and observe the varied HTML content

import requests

# Try replacing the URL with another data science-related page
url = "https://en.wikipedia.org/wiki/Web_scraping"   # Change this to any URL you want
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

print(f"Status Code   : {response.status_code}")
print(f"Content-Type  : {response.headers.get('Content-Type')}")
print("\nHTML Preview (First 500 characters):")
print(response.text[:500])

Status Code   : 200
Content-Type  : text/html; charset=UTF-8

HTML Preview (First 500 characters):
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp


In [5]:
# PROCEDURE 2: Parse HTML Content

import requests
from bs4 import BeautifulSoup

# Step 1: Define the target URL
# Using Wikipedia because towardsdatascience.com returns a 403 Forbidden error
url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {"User-Agent": "Mozilla/5.0"}

# Step 2: Send the GET request and raise error if request fails
response = requests.get(url, headers=headers)
response.raise_for_status()
html_content = response.text

# Step 3: Transform raw HTML into a BeautifulSoup object using the lxml parser
soup = BeautifulSoup(html_content, "lxml")

# Step 4: Confirm successful parsing by accessing the page title
print("--- Parsing HTML Content ---")
if soup.title:
    print(f"Parsed Page Title: {soup.title.text.strip()}")
else:
    print("Page title not found. HTML parsing might need adjustment.")
print("---")

--- Parsing HTML Content ---
Parsed Page Title: Wikipedia, the free encyclopedia
---


In [6]:
# TRY IT — Procedure 2: Examine the soup object structure

import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.text, "lxml")

# Try accessing different elements to explore the soup object
print("Page Title      :", soup.title.text.strip())
print("First <h1> tag  :", soup.find("h1").text.strip() if soup.find("h1") else "Not found")
print("First <p> tag   :", soup.find("p").text.strip()[:150] if soup.find("p") else "Not found")
print("\nType of soup    :", type(soup))
print("Type of title   :", type(soup.title))

Page Title      : Wikipedia, the free encyclopedia
First <h1> tag  : Main Page
First <p> tag   : Ornithoprion is an extinct genus of cartilaginous fish. Its only species lived during the Moscovian age. Its fossils are preserved in black shales fro

Type of soup    : <class 'bs4.BeautifulSoup'>
Type of title   : <class 'bs4.element.Tag'>


In [7]:
# PROCEDURE 3: Locate and Extract Specific Data Elements

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    article_links_data = []

    # Step 1: Target the "In the news" section (div with id='mp-itn')
    in_the_news_div = soup.find("div", id="mp-itn")

    if in_the_news_div:
        print("\n--- From 'In the news' section ---")
        links = in_the_news_div.find_all("a", href=True)
        for link in links:
            title_text = link.text.strip()
            href = link["href"]
            # Filter out non-article links (avoid Help:, Category: special pages)
            if title_text and href.startswith("/wiki/") and ":" not in href:
                absolute_url = urljoin(url, href)
                if not any(data["url"] == absolute_url for data in article_links_data):
                    article_links_data.append({"title": title_text, "url": absolute_url})

    # Step 2: Target the "On this day" section (div with id='mp-otd')
    on_this_day_div = soup.find("div", id="mp-otd")

    if on_this_day_div:
        print("\n--- From 'On this day' section ---")
        links = on_this_day_div.find_all("a", href=True)
        for link in links:
            title_text = link.text.strip()
            href = link["href"]
            if title_text and href.startswith("/wiki/") and ":" not in href:
                absolute_url = urljoin(url, href)
                if not any(data["url"] == absolute_url for data in article_links_data):
                    article_links_data.append({"title": title_text, "url": absolute_url})

    # Step 3: Display results
    if article_links_data:
        print("\nExtracted Article Titles and URLs (First 10):")
        for i, article in enumerate(article_links_data[:10]):
            print(f"- Title: '{article['title']}'")
            print(f"  URL  : {article['url']}")
    else:
        print("\nNo article links found in the targeted sections.")

except requests.exceptions.RequestException as e:
    print(f"\nError fetching URL {url}: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

print("---")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page



--- From 'In the news' section ---

--- From 'On this day' section ---

Extracted Article Titles and URLs (First 10):
- Title: 'Progressive Bulgaria'
  URL  : https://en.wikipedia.org/wiki/Progressive_Bulgaria
- Title: 'Rumen Radev'
  URL  : https://en.wikipedia.org/wiki/Rumen_Radev
- Title: 'National Assembly'
  URL  : https://en.wikipedia.org/wiki/National_Assembly_(Bulgaria)
- Title: 'the parliamentary elections'
  URL  : https://en.wikipedia.org/wiki/2026_Bulgarian_parliamentary_election
- Title: 'the ongoing Bulgarian political crisis'
  URL  : https://en.wikipedia.org/wiki/2021%E2%80%93present_Bulgarian_political_crisis
- Title: 'in Siverek'
  URL  : https://en.wikipedia.org/wiki/2026_Siverek_school_shooting
- Title: 'in Onikişubat'
  URL  : https://en.wikipedia.org/wiki/2026_Oniki%C5%9Fubat_school_shooting
- Title: 'Romuald Wadagni'
  URL  : https://en.wikipedia.org/wiki/Romuald_Wadagni
- Title: 'the Beninese presidential election'
  URL  : https://en.wikipedia.org/wiki/2026_Be

In [8]:
# PROCEDURE 3: Locate and Extract Specific Data Elements

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    article_links_data = []

    # Step 1: Target the "In the news" section (div with id='mp-itn')
    in_the_news_div = soup.find("div", id="mp-itn")

    if in_the_news_div:
        print("\n--- From 'In the news' section ---")
        links = in_the_news_div.find_all("a", href=True)
        for link in links:
            title_text = link.text.strip()
            href = link["href"]
            # Filter out non-article links (avoid Help:, Category: special pages)
            if title_text and href.startswith("/wiki/") and ":" not in href:
                absolute_url = urljoin(url, href)
                if not any(data["url"] == absolute_url for data in article_links_data):
                    article_links_data.append({"title": title_text, "url": absolute_url})

    # Step 2: Target the "On this day" section (div with id='mp-otd')
    on_this_day_div = soup.find("div", id="mp-otd")

    if on_this_day_div:
        print("\n--- From 'On this day' section ---")
        links = on_this_day_div.find_all("a", href=True)
        for link in links:
            title_text = link.text.strip()
            href = link["href"]
            if title_text and href.startswith("/wiki/") and ":" not in href:
                absolute_url = urljoin(url, href)
                if not any(data["url"] == absolute_url for data in article_links_data):
                    article_links_data.append({"title": title_text, "url": absolute_url})

    # Step 3: Display results
    if article_links_data:
        print("\nExtracted Article Titles and URLs (First 10):")
        for i, article in enumerate(article_links_data[:10]):
            print(f"- Title: '{article['title']}'")
            print(f"  URL  : {article['url']}")
    else:
        print("\nNo article links found in the targeted sections.")

except requests.exceptions.RequestException as e:
    print(f"\nError fetching URL {url}: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

print("---")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page

--- From 'In the news' section ---

--- From 'On this day' section ---

Extracted Article Titles and URLs (First 10):
- Title: 'Progressive Bulgaria'
  URL  : https://en.wikipedia.org/wiki/Progressive_Bulgaria
- Title: 'Rumen Radev'
  URL  : https://en.wikipedia.org/wiki/Rumen_Radev
- Title: 'National Assembly'
  URL  : https://en.wikipedia.org/wiki/National_Assembly_(Bulgaria)
- Title: 'the parliamentary elections'
  URL  : https://en.wikipedia.org/wiki/2026_Bulgarian_parliamentary_election
- Title: 'the ongoing Bulgarian political crisis'
  URL  : https://en.wikipedia.org/wiki/2021%E2%80%93present_Bulgarian_political_crisis
- Title: 'in Siverek'
  URL  : https://en.wikipedia.org/wiki/2026_Siverek_school_shooting
- Title: 'in Onikişubat'
  URL  : https://en.wikipedia.org/wiki/2026_Oniki%C5%9Fubat_school_shooting
- Title: 'Romuald Wadagni'
  URL  : https://en.wikipedia.org/wiki/Romuald_Wadagni
- Title: 'the Beninese preside

In [9]:
# PROCEDURE 3: Locate and Extract Specific Data Elements

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    article_links_data = []

    # Step 1: Target the "In the news" section (div with id='mp-itn')
    in_the_news_div = soup.find("div", id="mp-itn")

    if in_the_news_div:
        print("\n--- From 'In the news' section ---")
        links = in_the_news_div.find_all("a", href=True)
        for link in links:
            title_text = link.text.strip()
            href = link["href"]
            # Filter out non-article links (avoid Help:, Category: special pages)
            if title_text and href.startswith("/wiki/") and ":" not in href:
                absolute_url = urljoin(url, href)
                if not any(data["url"] == absolute_url for data in article_links_data):
                    article_links_data.append({"title": title_text, "url": absolute_url})

    # Step 2: Target the "On this day" section (div with id='mp-otd')
    on_this_day_div = soup.find("div", id="mp-otd")

    if on_this_day_div:
        print("\n--- From 'On this day' section ---")
        links = on_this_day_div.find_all("a", href=True)
        for link in links:
            title_text = link.text.strip()
            href = link["href"]
            if title_text and href.startswith("/wiki/") and ":" not in href:
                absolute_url = urljoin(url, href)
                if not any(data["url"] == absolute_url for data in article_links_data):
                    article_links_data.append({"title": title_text, "url": absolute_url})

    # Step 3: Display results
    if article_links_data:
        print("\nExtracted Article Titles and URLs (First 10):")
        for i, article in enumerate(article_links_data[:10]):
            print(f"- Title: '{article['title']}'")
            print(f"  URL  : {article['url']}")
    else:
        print("\nNo article links found in the targeted sections.")

except requests.exceptions.RequestException as e:
    print(f"\nError fetching URL {url}: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

print("---")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page

--- From 'In the news' section ---

--- From 'On this day' section ---

Extracted Article Titles and URLs (First 10):
- Title: 'Progressive Bulgaria'
  URL  : https://en.wikipedia.org/wiki/Progressive_Bulgaria
- Title: 'Rumen Radev'
  URL  : https://en.wikipedia.org/wiki/Rumen_Radev
- Title: 'National Assembly'
  URL  : https://en.wikipedia.org/wiki/National_Assembly_(Bulgaria)
- Title: 'the parliamentary elections'
  URL  : https://en.wikipedia.org/wiki/2026_Bulgarian_parliamentary_election
- Title: 'the ongoing Bulgarian political crisis'
  URL  : https://en.wikipedia.org/wiki/2021%E2%80%93present_Bulgarian_political_crisis
- Title: 'in Siverek'
  URL  : https://en.wikipedia.org/wiki/2026_Siverek_school_shooting
- Title: 'in Onikişubat'
  URL  : https://en.wikipedia.org/wiki/2026_Oniki%C5%9Fubat_school_shooting
- Title: 'Romuald Wadagni'
  URL  : https://en.wikipedia.org/wiki/Romuald_Wadagni
- Title: 'the Beninese preside

In [21]:
# TRY IT — Procedure 3: Extract a specific element using browser developer tools

import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")

    # Try It 1: Extract the article's main title
    main_title = soup.find("h1", id="firstHeading")
    print("Main Title      :", main_title.text.strip() if main_title else "Not found")

    # Try It 2: Extract the introductory paragraph
    content_div = soup.find("div", id="mw-content-text")
    if content_div:
        for p in content_div.find_all("p"):
            text = p.text.strip()
            if text:
                print("First Paragraph :", text[:200], "...")
                break

    # Try It 3: Extract the last modified date from the footer
    last_mod = soup.find("li", id="footer-info-lastmod")
    print("Last Modified   :", last_mod.text.strip() if last_mod else "Not found")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Main Title      : Artificial intelligence
First Paragraph : Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and dec ...
Last Modified   : This page was last edited on 22 April 2026, at 04:06 (UTC).


In [10]:
# PROCEDURE 4: Extract Data from HTML Tables

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://en.wikipedia.org/wiki/List_of_datasets_for_machine_learning_research"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.text, "lxml")

print("--- PROCEDURE 4: Extract Data from HTML Tables ---")

# Step 1: Identify the specific table to extract
target_table = soup.find("table", class_="wikitable")

if target_table:
    # Step 2: Extract table headers (column names)
    table_headers = [th.text.strip() for th in target_table.find("tr").find_all("th")]
    print(f"Table Headers (First 5): {table_headers[:5]}")

    # Step 3: Extract data rows, limiting to first 5 for demo
    table_data = []
    for row in target_table.find_all("tr")[1:6]:
        cells     = row.find_all(["td", "th"])
        cell_texts = [cell.text.strip() for cell in cells]
        table_data.append(cell_texts)

    print("\nFirst 5 Data Rows (Raw Lists):")
    for i, row in enumerate(table_data):
        print(f"Row {i+1}: {row[:5]}...")

    # Step 4: Convert data into a Pandas DataFrame
    if table_data:
        try:
            df = pd.DataFrame(table_data, columns=table_headers[:len(table_data[0])])
            print("\nDataFrame Representation (First 3 rows):")
            print(df.head(3).to_string())
        except ValueError as e:
            print(f"\nCould not create DataFrame. Column count mismatch: {e}")
    else:
        print("No data rows found in the table.")
else:
    print("Target table with class 'wikitable' not found on the page.")

print("---")

--- PROCEDURE 4: Extract Data from HTML Tables ---
Table Headers (First 5): ['Type', 'Subtypes']

First 5 Data Rows (Raw Lists):
Row 1: ['Specific category', 'Finance, Economics, Commerce, Societal, Health, Academy, Sports, Food, Agriculture, Travel, Geospatial, Political, Consumer, Transport,  Logistics, Environmental, Real-Estate, Legal, Entertainment, Energy, Hospitality']...
Row 2: ['Scope', 'Supranational Union, National, Subnational, Municipality, Urban, Rural']...
Row 3: ['Language', 'Mandarin Chinese, Spanish, English, Arabic, Hindi, Bengali']...
Row 4: ['Type', 'Tabular, Graph, Text, Image, Sound, Video']...
Row 5: ['Usage', 'Training, validating, and testing']...

DataFrame Representation (First 3 rows):
                Type                                                                                                                                                                                                                   Subtypes
0  Specific category  Finance, Econo

In [11]:
# TRY IT — Procedure 4: Extract a different Wikipedia table

import requests
from bs4 import BeautifulSoup
import pandas as pd

# Try changing this URL to any Wikipedia page with a table
url = "https://en.wikipedia.org/wiki/Comparison_of_programming_languages"
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")

    table = soup.find("table", class_="wikitable")
    if table:
        headers_list = [th.text.strip() for th in table.find("tr").find_all("th")]
        print(f"Headers found  : {headers_list[:6]}")

        rows = []
        for row in table.find_all("tr")[1:6]:
            cells = row.find_all(["td", "th"])
            rows.append([c.text.strip() for c in cells])

        df = pd.DataFrame(rows, columns=headers_list[:len(rows[0])] if rows else headers_list)
        print("\nFirst 5 rows:")
        print(df.head().to_string())
    else:
        print("No wikitable found on this page.")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Headers found  : ['Language', 'Original purpose', 'Imperative', 'Object-oriented', 'Functional', 'Procedural']

First 5 rows:
                             Language                                  Original purpose Imperative Object-oriented Functional Procedural Generic Reflective                            Other paradigms                                                                                                                                          Standardized
0  1C:Enterprise programming language  Application, RAD, business, general, web, mobile        Yes              No        Yes        Yes     Yes        Yes  Object-based, Prototype-based programming                                                                                                                                                    No
1                        ActionScript                     Application, client-side, web        Yes             Yes        Yes        Yes      No         No                     

In [12]:
# PROCEDURE 5: Handle Pagination

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

print("--- PROCEDURE 5: Handle Pagination (Quotes to Scrape) ---")

start_url      = "http://quotes.toscrape.com/"
all_quotes     = []
pages_to_collect = 3       # Limiting to 3 pages for demonstration
current_page_url = start_url
page_counter   = 0

while page_counter < pages_to_collect:
    print(f"\nAttempting to fetch page {page_counter + 1}: {current_page_url}")
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        response = requests.get(current_page_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")

        # Step 1: Extract quotes from the current page
        quote_elements = soup.find_all("div", class_="quote")

        if quote_elements:
            for quote_item in quote_elements:
                text   = quote_item.find("span", class_="text").text.strip()
                author = quote_item.find("small", class_="author").text.strip()
                all_quotes.append({"text": text, "author": author})
            print(f"Collected {len(quote_elements)} quotes from this page.")
        else:
            print("No quote elements found on this page. Stopping.")
            break

        # Step 2: Locate the "Next Page" link
        next_li = soup.find("li", class_="next")

        if next_li:
            next_link_element = next_li.find("a", href=True)
            if next_link_element:
                next_page_relative_url = next_link_element["href"]
                current_page_url = urljoin(start_url, next_page_relative_url)
                page_counter += 1
                time.sleep(1)   # Be polite: pause before fetching the next page
            else:
                print("Next page link not found. Stopping.")
                break
        else:
            print("No 'next' pagination link found. Assuming last page.")
            break

    except requests.exceptions.RequestException as e:
        print(f"ERROR (Network/HTTP): {e}")
        break
    except Exception as e:
        print(f"ERROR (Unexpected): {e}")
        break

print(f"\n--- Total Quotes Collected Across {page_counter} pages: {len(all_quotes)} ---")
print("First 5 Collected Quotes (Sample):")
for i, quote in enumerate(all_quotes[:5]):
    print(f"- \"{quote['text']}\" - {quote['author']}")
print("--- End of Procedure ---")

--- PROCEDURE 5: Handle Pagination (Quotes to Scrape) ---

Attempting to fetch page 1: http://quotes.toscrape.com/


Collected 10 quotes from this page.

Attempting to fetch page 2: http://quotes.toscrape.com/page/2/
Collected 10 quotes from this page.

Attempting to fetch page 3: http://quotes.toscrape.com/page/3/
Collected 10 quotes from this page.

--- Total Quotes Collected Across 3 pages: 30 ---
First 5 Collected Quotes (Sample):
- "“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”" - Albert Einstein
- "“It is our choices, Harry, that show what we truly are, far more than our abilities.”" - J.K. Rowling
- "“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”" - Albert Einstein
- "“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”" - Jane Austen
- "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”" - Marilyn Monroe
--- End of Procedure ---


In [13]:
# TRY IT — Procedure 5: Paginate through books.toscrape.com

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

# Inspect http://books.toscrape.com/ and try to scrape book titles across pages
start_url        = "http://books.toscrape.com/"
all_books        = []
pages_to_collect = 3
current_url      = start_url
page_counter     = 0

headers = {"User-Agent": "Mozilla/5.0"}

while page_counter < pages_to_collect:
    print(f"\nFetching page {page_counter + 1}: {current_url}")
    try:
        response = requests.get(current_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")

        # Books are inside <article class="product_pod">
        books = soup.find_all("article", class_="product_pod")
        for book in books:
            title = book.find("h3").find("a")["title"]
            price = book.find("p", class_="price_color").text.strip()
            all_books.append({"title": title, "price": price})

        print(f"Collected {len(books)} books from this page.")

        # Find the next page link
        next_btn = soup.find("li", class_="next")
        if next_btn:
            next_url     = next_btn.find("a")["href"]
            current_url  = urljoin(current_url, next_url)
            page_counter += 1
            time.sleep(1)
        else:
            print("No next page found.")
            break

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        break

print(f"\nTotal books collected: {len(all_books)}")
for book in all_books[:5]:
    print(f"- {book['title']} | {book['price']}")


Fetching page 1: http://books.toscrape.com/
Collected 20 books from this page.

Fetching page 2: http://books.toscrape.com/catalogue/page-2.html
Collected 20 books from this page.

Fetching page 3: http://books.toscrape.com/catalogue/page-3.html
Collected 20 books from this page.

Total books collected: 60
- A Light in the Attic | Â£51.77
- Tipping the Velvet | Â£53.74
- Soumission | Â£50.10
- Sharp Objects | Â£47.82
- Sapiens: A Brief History of Humankind | Â£54.23


In [14]:
# PROCEDURE 6: Clean and Standardize Extracted Data

import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url     = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {"User-Agent": "Mozilla/5.0"}

print("--- PROCEDURE 6: Clean and Standardize Extracted Data ---")

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup  = BeautifulSoup(response.text, "lxml")
    table = soup.find("table", class_="wikitable")

    if table:
        # Step 1: Extract raw headers
        header_cells  = table.find("tr").find_all("th")
        headers_text  = [th.text.strip() for th in header_cells]

        # Step 2: Locate required column indices
        software_idx        = headers_text.index("Software")        if "Software"        in headers_text else -1
        creator_idx         = headers_text.index("Creator")         if "Creator"         in headers_text else -1
        initial_release_idx = headers_text.index("Initial release") if "Initial release" in headers_text else -1

        required_indices = [software_idx, creator_idx, initial_release_idx]
        required_names   = ["Software", "Creator", "Initial release"]

        if any(idx == -1 for idx in required_indices):
            print("ERROR: One or more required columns are missing.")
            print(f"Expected : {required_names}")
            print(f"Found    : {headers_text}")
        else:
            cleaned_data_records = []
            data_rows = table.find_all("tr")[1:]

            # Step 3: Extract and clean each row
            for row_num, row in enumerate(data_rows):
                cols = row.find_all("td")
                if len(cols) > max(required_indices):
                    software_name      = cols[software_idx].text.strip()
                    creator_name       = cols[creator_idx].text.strip()
                    raw_initial_release = cols[initial_release_idx].text.strip()
                    # Step 4: Remove bracketed references using regex
                    clean_initial_release = re.sub(r"\[.*?\]", "", raw_initial_release).strip()
                    cleaned_data_records.append({
                        "Software"            : software_name,
                        "Creator"             : creator_name,
                        "Initial_Release_Date": clean_initial_release
                    })
                else:
                    print(f"Skipping row {row_num + 1}: insufficient columns.")

            print("Cleaned and Standardized Data (First 5 records):")
            for record in cleaned_data_records[:5]:
                print(record)

            df_cleaned = pd.DataFrame(cleaned_data_records)
            print("\nDataFrame Info after Cleaning:")
            df_cleaned.info()
            print("\nDataFrame Head:")
            print(df_cleaned.head().to_string())
    else:
        print("Table with class 'wikitable' not found.")

except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): {e}")
except Exception as e:
    print(f"ERROR (Unexpected): {e}")
finally:
    print("--- End of Procedure ---")

--- PROCEDURE 6: Clean and Standardize Extracted Data ---
Skipping row 31: insufficient columns.
Cleaned and Standardized Data (First 5 records):
{'Software': 'BigDL', 'Creator': 'Jason Dai (Intel)', 'Initial_Release_Date': '2016'}
{'Software': 'Caffe', 'Creator': 'Berkeley Vision and Learning Center', 'Initial_Release_Date': '2013'}
{'Software': 'Chainer', 'Creator': 'Preferred Networks', 'Initial_Release_Date': '2015'}
{'Software': 'Deeplearning4j', 'Creator': 'Skymind engineering team; Deeplearning4j community; originally Adam Gibson', 'Initial_Release_Date': '2014'}
{'Software': 'DeepSpeed', 'Creator': 'Microsoft', 'Initial_Release_Date': '2019'}

DataFrame Info after Cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Software              30 non-null     str  
 1   Creator               30 non-null     str  
 2   Initial_Release_D

In [16]:
# TRY IT — Procedure 6: Clean a population column with commas and citation brackets

import requests
from bs4 import BeautifulSoup
import re

url     = "https://en.wikipedia.org/wiki/List_of_countries_by_population_(2023)"
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup  = BeautifulSoup(response.text, "lxml")
    table = soup.find("table", class_="wikitable")

    if table:
        header_cells = table.find("tr").find_all("th")
        headers_text = [th.text.strip() for th in header_cells]

        # Find the population column index
        pop_idx = next((i for i, h in enumerate(headers_text) if "Population" in h), -1)

        if pop_idx != -1:
            print(f"Population column found at index: {pop_idx}")
            for row in table.find_all("tr")[1:6]:
                cols = row.find_all("td")
                if len(cols) > pop_idx:
                    raw_pop   = cols[pop_idx].text.strip()
                    # Remove commas and citation brackets, then convert to int
                    clean_pop = re.sub(r"\[.*?\]", "", raw_pop).replace(",", "").strip()
                    try:
                        pop_int = int(clean_pop)
                        print(f"Raw: '{raw_pop}'  -->  Cleaned: {pop_int:,}")
                    except ValueError:
                        print(f"Raw: '{raw_pop}'  -->  Could not convert to int: '{clean_pop}'")
        else:
            print("Population column not found.")
    else:
        print("No wikitable found on this page.")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Error: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/List_of_countries_by_population_(2023)


In [15]:
# PROCEDURE 7: Implement Error Handling and Robustness

import requests
from bs4 import BeautifulSoup

print("--- PROCEDURE 7: Implement Error Handling and Robustness ---")

# Test URLs designed to trigger different error types
test_scenarios = {
    "valid_article_page"     : "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "non_existent_article"   : "http://httpbin.org/status/404",        # Expected 404 HTTP Error
    "slow_server_sim"        : "http://httpbin.org/delay/6",            # Causes timeout with timeout=5
    "invalid_domain_connect" : "http://this.domain.does.not.resolve.abc",  # Connection error
}

for scenario_name, test_url in test_scenarios.items():
    print(f"\nTesting Scenario: '{scenario_name}' at {test_url}")
    try:
        headers  = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(test_url, headers=headers, timeout=5)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        if scenario_name == "valid_article_page":
            main_title_element = soup.find("h1", id="firstHeading")
        else:
            main_title_element = soup.find("h1")

        if main_title_element:
            print(f"SUCCESS: Found title: '{main_title_element.text.strip()[:100]}...'")
        else:
            print("SUCCESS (with caveat): Fetched page, but main title element not found.")

    except requests.exceptions.HTTPError as e:
        print(f"ERROR (HTTP): Status Code {e.response.status_code}. Reason: {e.response.reason}")
    except requests.exceptions.ConnectionError as e:
        print(f"ERROR (Connection): Could not connect to {test_url}.")
    except requests.exceptions.Timeout as e:
        print(f"ERROR (Timeout): Request to {test_url} timed out after 5 seconds.")
    except requests.exceptions.RequestException as e:
        print(f"ERROR (General Request): {e}")
    except Exception as e:
        print(f"ERROR (Unexpected): {e}")

print("---")

--- PROCEDURE 7: Implement Error Handling and Robustness ---

Testing Scenario: 'valid_article_page' at https://en.wikipedia.org/wiki/Artificial_intelligence
SUCCESS: Found title: 'Artificial intelligence...'

Testing Scenario: 'non_existent_article' at http://httpbin.org/status/404
ERROR (HTTP): Status Code 404. Reason: NOT FOUND

Testing Scenario: 'slow_server_sim' at http://httpbin.org/delay/6
ERROR (Timeout): Request to http://httpbin.org/delay/6 timed out after 5 seconds.

Testing Scenario: 'invalid_domain_connect' at http://this.domain.does.not.resolve.abc
ERROR (Connection): Could not connect to http://this.domain.does.not.resolve.abc.
---


In [17]:
# TRY IT — Procedure 7: Experiment with timeout values

import requests

# Try It 1: Set timeout very HIGH — server responds before timeout
url     = "http://httpbin.org/delay/6"
headers = {"User-Agent": "Mozilla/5.0"}

print("Test 1 — High timeout (30s):")
try:
    response = requests.get(url, headers=headers, timeout=30)  # Change to 30
    print(f"SUCCESS: Status {response.status_code} — server responded in time.")
except requests.exceptions.Timeout:
    print("ERROR (Timeout): Still timed out even with high timeout.")
except requests.exceptions.RequestException as e:
    print(f"ERROR: {e}")

# Try It 2: Set timeout very LOW — force a timeout
print("\nTest 2 — Low timeout (1s):")
try:
    response = requests.get(url, headers=headers, timeout=1)   # Change to 1
    print(f"SUCCESS (unexpected): {response.status_code}")
except requests.exceptions.Timeout:
    print("ERROR (Timeout): Request timed out after 1 second as expected.")
except requests.exceptions.RequestException as e:
    print(f"ERROR: {e}")

Test 1 — High timeout (30s):


SUCCESS: Status 200 — server responded in time.

Test 2 — Low timeout (1s):
ERROR (Timeout): Request timed out after 1 second as expected.


In [18]:
# TRY IT — Procedure 7: Experiment with timeout values

import requests

# Try It 1: Set timeout very HIGH — server responds before timeout
url     = "http://httpbin.org/delay/6"
headers = {"User-Agent": "Mozilla/5.0"}

print("Test 1 — High timeout (30s):")
try:
    response = requests.get(url, headers=headers, timeout=30)  # Change to 30
    print(f"SUCCESS: Status {response.status_code} — server responded in time.")
except requests.exceptions.Timeout:
    print("ERROR (Timeout): Still timed out even with high timeout.")
except requests.exceptions.RequestException as e:
    print(f"ERROR: {e}")

# Try It 2: Set timeout very LOW — force a timeout
print("\nTest 2 — Low timeout (1s):")
try:
    response = requests.get(url, headers=headers, timeout=1)   # Change to 1
    print(f"SUCCESS (unexpected): {response.status_code}")
except requests.exceptions.Timeout:
    print("ERROR (Timeout): Request timed out after 1 second as expected.")
except requests.exceptions.RequestException as e:
    print(f"ERROR: {e}")

Test 1 — High timeout (30s):
SUCCESS: Status 200 — server responded in time.

Test 2 — Low timeout (1s):
ERROR (Timeout): Request timed out after 1 second as expected.


In [19]:
# PROCEDURE 8: Store the Extracted Data (CSV, JSON, SQLite)

import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os
import re
import sqlite3

print("\n--- PROCEDURE 8: Store the Extracted Data ---")

url     = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup  = BeautifulSoup(response.text, "lxml")
    final_data_to_save = []
    table = soup.find("table", class_="wikitable")

    if table:
        # Step 1: Extract and standardize headers
        table_headers_raw = [th.text.strip() for th in table.find("tr").find_all("th")]
        table_headers     = [
            re.sub(r"\[.*?\]|\s\(.*?\)", "", h).replace(" ", "_").strip()
            for h in table_headers_raw
        ]

        data_rows = table.find_all("tr")[1:]

        # Step 2: Extract up to 15 rows of data
        for i, row in enumerate(data_rows[:15]):
            cols     = row.find_all("td")
            row_data = {}
            for j, col in enumerate(cols):
                if j < len(table_headers):
                    row_data[table_headers[j]] = col.text.strip()
            if row_data:
                final_data_to_save.append(row_data)
    else:
        print("ERROR: Table with class 'wikitable' not found.")

    if final_data_to_save:
        df_output = pd.DataFrame(final_data_to_save)
        print("\n--- Data Ready for Archiving (DataFrame Head) ---")
        print(df_output.head().to_string())

        # Step 3: Save as CSV
        csv_file_path = "deep_learning_software_comparison.csv"
        df_output.to_csv(csv_file_path, index=False, encoding="utf-8")
        print(f"\nSaved to: {csv_file_path} (CSV format)")

        # Step 4: Save as JSON
        json_file_path = "deep_learning_software_comparison.json"
        df_output.to_json(json_file_path, orient="records", indent=4)
        print(f"Saved to: {json_file_path} (JSON format)")

        # Step 5: Save to SQLite database
        db_file_path = "deep_learning_software_comparison.db"
        table_name   = "dl_software_comparison"
        conn         = None
        try:
            conn = sqlite3.connect(db_file_path)
            df_output.to_sql(table_name, conn, if_exists="replace", index=False)
            print(f"Saved to: {db_file_path} (SQLite database, table: '{table_name}')")
        except Exception as db_err:
            print(f"ERROR (SQLite): {db_err}")
        finally:
            if conn:
                conn.close()

        # Step 6: Verify all files were created
        all_exist = (
            os.path.exists(csv_file_path) and
            os.path.exists(json_file_path) and
            os.path.exists(db_file_path)
        )
        if all_exist:
            print("\nVerification: CSV, JSON, and SQLite files are all present.")
        else:
            print("\nVerification Failed: One or more files were not found.")
    else:
        print("No data collected to proceed with storage.")

except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): {e}")
except Exception as e:
    print(f"ERROR (Unexpected): {e}")


--- PROCEDURE 8: Store the Extracted Data ---

--- Data Ready for Archiving (DataFrame Head) ---
         Software                                                                     Creator Initial_release Software_license Open_source                                         Platform         Written_in                                     Interface OpenMP_support        OpenCL_support CUDA_support ROCm_support Automatic_differentiation Has_pretrained_models Recurrent_nets Convolutional_nets RBM/DBNs Parallel_execution(multi_node) Actively_developed
0           BigDL                                                           Jason Dai (Intel)            2016       Apache 2.0         Yes                                     Apache Spark              Scala                                 Scala, Python                                                No           No                                             Yes            Yes                Yes                                                

In [20]:
# TRY IT — Procedure 8: Open and inspect the generated JSON file

import json

# Open and print the JSON file generated in Procedure 8
with open("deep_learning_software_comparison.json") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

[
  {
    "Software": "BigDL",
    "Creator": "Jason Dai (Intel)",
    "Initial_release": "2016",
    "Software_license": "Apache 2.0",
    "Open_source": "Yes",
    "Platform": "Apache Spark",
    "Written_in": "Scala",
    "Interface": "Scala, Python",
    "OpenMP_support": "",
    "OpenCL_support": "",
    "CUDA_support": "No",
    "ROCm_support": "No",
    "Automatic_differentiation": "",
    "Has_pretrained_models": "Yes",
    "Recurrent_nets": "Yes",
    "Convolutional_nets": "Yes",
    "RBM/DBNs": "",
    "Parallel_execution(multi_node)": "",
    "Actively_developed": "Yes"
  },
  {
    "Software": "Caffe",
    "Creator": "Berkeley Vision and Learning Center",
    "Initial_release": "2013",
    "Software_license": "BSD",
    "Open_source": "Yes",
    "Platform": "Linux, macOS, Windows[3]",
    "Written_in": "C++",
    "Interface": "Python, MATLAB, C++",
    "OpenMP_support": "Yes",
    "OpenCL_support": "Under development[4]",
    "CUDA_support": "Yes",
    "ROCm_support": "No",

**THE END OF LAB 9 PROCEDURES**

**DATA AND OBSERVATION: CODE SNIPPET**

In [ ]:
# Snippet 1
import requests

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=5)
    print(f"HTTP Status Code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")

: 

In [ ]:
# Snippet 2
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    print(f"Page Title (from <title> tag): {soup.title.text}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

: 

In [ ]:
# Snippet 3
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    main_heading = soup.find("h1", id="firstHeading")
    if main_heading:
        print(f"Main Heading: '{main_heading.text.strip()}'")
    else:
        print("Main heading (h1 with id='firstHeading') not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

: 

In [ ]:
# Snippet 4
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    content_div = soup.find("div", id="mw-content-text")
    first_paragraph = None
    if content_div:
        paragraphs = content_div.find_all("p")
        for p in paragraphs:
            text = p.text.strip()
            if text and not p.find_parent("table", class_="infobox"):
                first_paragraph = text
                break
    if first_paragraph:
        print(f"First Paragraph (truncated): '{first_paragraph[:150]}...'")
    else:
        print("First meaningful paragraph not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

: 

In [ ]:
# Snippet 5
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    first_quote_div = soup.find("div", class_="quote")
    if first_quote_div:
        quote_text_element = first_quote_div.find("span", class_="text")
        author_element = first_quote_div.find("small", class_="author")
        if quote_text_element and author_element:
            quote = quote_text_element.text.strip()
            author = author_element.text.strip()
            print(f"Sample Quote: '{quote}'")
            print(f"Sample Author: {author}")
        else:
            print("No usable quote text or author element found within the first quote div.")
    else:
        print("No quote div found. The website structure may have changed, or a different selector is needed.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

: 

In [ ]:
# Snippet 6
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    first_quote_div = soup.find("div", class_="quote")
    if first_quote_div:
        author_element = first_quote_div.find("small", class_="author")
        if author_element:
            author_name = author_element.text.strip()
            print(f"Author of the first quote: '{author_name}'")
        else:
            print("Author element not found within the first quote div (class='author').")
    else:
        print("First quote container (div with class 'quote') not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

: 

In [ ]:
# Snippet 7
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    all_quote_divs = soup.find_all("div", class_="quote")
    if all_quote_divs:
        print("Authors found on the page:")
        authors_list = []
        for i, quote_div in enumerate(all_quote_divs):
            author_element = quote_div.find("small", class_="author")
            if author_element:
                author_name = author_element.text.strip()
                authors_list.append(author_name)
                print(f"- {author_name}")
            else:
                print(f"- Author not found for quote #{i+1}.")
        print(f"\nTotal unique authors found: {len(set(authors_list))}")
    else:
        print("No quote containers (divs with class 'quote') found on the page.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

: 

In [ ]:
# Snippet 8
import requests
from bs4 import BeautifulSoup
import re

url = "https://en.wikipedia.org/wiki/List_of_countries_by_population"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    table = soup.find("table", class_="wikitable")
    if table:
        header_cells = table.find("tr").find_all("th")
        headers_text = [th.text.strip() for th in header_cells]
        population_idx = -1
        for i, header in enumerate(headers_text):
            if "Population" in header:
                population_idx = i
                break
        if population_idx != -1:
            first_data_row = table.find_all("tr")[1]
            cols = first_data_row.find_all("td")
            if len(cols) > population_idx:
                raw_population = cols[population_idx].text.strip()
                print(f"Raw Population String (e.g., World population): '{raw_population}'")
            else:
                print("Population column not found in the first data row (index out of bounds).")
        else:
            print("Population header not found in table.")
    else:
        print("Table with class 'wikitable' not found on the page.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

: 

In [ ]:
# Snippet 9
import pandas as pd

items = soup.select(".thumbnail")
data = []
for it in items:
    data.append(
        {
            "name": it.select_one(".title")["title"],
            "price": it.select_one(".price").text,
        }
    )
df = pd.DataFrame(data)
df.to_csv("laptops.csv", index=False)
print("Saved laptops.csv")

: 

In [ ]:
# Snippet 10
import requests

url = "http://httpbin.org/delay/6"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    print("SUCCESS: Page fetched (unexpected for timeout).")
except requests.exceptions.Timeout as e:
    print(f"ERROR (Timeout): Request to {url} timed out after [X] seconds. Details: {e}")
except requests.exceptions.RequestException as e:
    print(f"ERROR (General Request): {e}")

: 

In [ ]:
# Snippet 11
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

url = "http://quotes.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "lxml")
all_quote_divs = soup.find_all("div", class_="quote")

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find("span", class_="text")
    author_element = quote_div.find("small", class_="author")
    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"
    scraped_data.append({"Quote": quote, "Author": author})

df_scraped = pd.DataFrame(scraped_data)
csv_file_path = "web_scraped_quotes.csv"
df_scraped.to_csv(csv_file_path, index=False, encoding="utf-8")
print(csv_file_path)

: 

In [ ]:
# Snippet 12
import requests
from bs4 import BeautifulSoup
import json
import os

url = "http://quotes.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "lxml")
all_quote_divs = soup.find_all("div", class_="quote")

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find("span", class_="text")
    author_element = quote_div.find("small", class_="author")
    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"
    scraped_data.append({"quote": quote, "author": author})

json_file_path = "web_scraped_quotes.json"
with open(json_file_path, "w", encoding="utf-8") as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=4)
print(json_file_path)

: 

In [ ]:
# Snippet 13
import requests
from bs4 import BeautifulSoup
import sqlite3
import os

url = "http://quotes.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "lxml")
all_quote_divs = soup.find_all("div", class_="quote")

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find("span", class_="text")
    author_element = quote_div.find("small", class_="author")
    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"
    scraped_data.append({"quote": quote, "author": author})

db_file_path = "web_scraped_quotes.db"
conn = sqlite3.connect(db_file_path)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS quotes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        quote TEXT NOT NULL,
        author TEXT NOT NULL
    )
""")

for item in scraped_data:
    cursor.execute(
        "INSERT INTO quotes (quote, author) VALUES (?, ?)",
        (item["quote"], item["author"]),
    )

conn.commit()
conn.close()
print(db_file_path)

: 

In [ ]:
# Snippet 14
import pandas as pd
import sqlite3
import os

db_file_path = "web_scraped_quotes.db"
table_name = "quotes"

if os.path.exists(db_file_path):
    conn = sqlite3.connect(db_file_path)
    df_read = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    print("\nData read from SQLite ('web_scraped_quotes.db'):")
    print(df_read.to_string())
    conn.close()
else:
    print(f"SQLite database '{db_file_path}' not found. Please ensure the data was stored to SQLite first.")

: 

In [ ]:
# Snippet 15
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    quote_elements = soup.find_all("div", class_="quote")

    if quote_elements:
        print(f"Found {len(quote_elements)} quotes on the first page.")
        for i, quote_item in enumerate(quote_elements[:2]):
            text = quote_item.find("span", class_="text").text.strip()
            author = quote_item.find("small", class_="author").text.strip()
            print(f'Quote {i+1}: "{text[:70]}..." - {author}')
    else:
        print("No quotes found on the first page.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

: 

In [ ]:
# Snippet 16
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "http://quotes.toscrape.com/"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    next_li = soup.find("li", class_="next")

    if next_li:
        next_link_element = next_li.find("a", href=True)
        if next_link_element:
            next_page_relative_url = next_link_element["href"]
            full_next_page_url = urljoin(url, next_page_relative_url)
            print(f"Found 'Next Page' link: {full_next_page_url}")
        else:
            print("Next page <a> tag not found within 'next' li.")
    else:
        print("No 'next' pagination li found (might be on the last page).")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

: 

In [ ]:
# Snippet 17
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    table = soup.find("table", class_="wikitable")
    if table:
        header_cells = table.find("tr").find_all("th")
        raw_headers = [th.text.strip() for th in header_cells]
        print(f"Raw Headers: {raw_headers}")
    else:
        print("Table not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

: 

In [ ]:
# Snippet 18
import requests
from bs4 import BeautifulSoup
import re

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "lxml")

table = soup.find("table", class_="wikitable")
header_cells = table.find("tr").find_all("th")
raw_headers = [th.text.strip() for th in header_cells]
print(f"Raw Headers from Web Scraping: {raw_headers}")

standardized_headers = [
    re.sub(r"\[.*?\]|\s\(.*?\)", "", h).replace(" ", "_").strip() for h in raw_headers
]
print(f"Standardized Headers: {standardized_headers}")

: 

In [ ]:
# Snippet 19
import requests

url = "http://this.domain.does.not.resolve.abc"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    print("SUCCESS (unexpected)")
except requests.exceptions.ConnectionError as e:
    print(f"ERROR (Connection): Could not establish connection to {url}. Details: {e}")
except requests.exceptions.RequestException as e:
    print(f"ERROR (General Request): {e}")

: 

In [ ]:
# Snippet 20
import requests
from bs4 import BeautifulSoup
import re

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {"User-Agent": "Mozilla/5.0"}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    last_modified_element = soup.find("li", id="footer-info-lastmod")
    if last_modified_element:
        raw_text = last_modified_element.text.strip()
        date_match = re.search(r"on (\d{1,2} \w+ \d{4})", raw_text)
        if date_match:
            clean_date = date_match.group(1)
            print(f"Last Modified Date: '{clean_date}'")
        else:
            print(f"Last Modified Date (raw, unable to parse): '{raw_text}'")
    else:
        print("Last modified element not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

: 

**THE END OF CODE SNIPPET FOR DATA AND OBSERVATION.**

# THE END OF LAB 9: AUTOMATING DATA COLLECTION (VNS)
# BY CRESENCIO
**LOVE YOU CHAEYOUNG**